In [48]:
!pip install chromadb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [49]:
# Create client/ server

import chromadb
client = chromadb.PersistentClient(path="./chroma_data")

# chromadb.Client() # normal in memory client
# chromadb.PersistentClient()  # for hard disk storage, documents available across runs
# chromadb.HttpClient()   # For connecting to server over network

In [50]:
# Read data and create chunks

from pypdf import PdfReader

reader = PdfReader("The indigenous badugar of the Nilgiris.pdf")

page_text = []
for page in reader.pages:
    page_text.append(page.extract_text())

complete_data = ""
for text in page_text:
    complete_data += text + "\n"

chunks = complete_data.split("\n")

print(chunks[:10])

['  ', '25 ', 'International Journal of Humanities and Social Science Research ', 'www.socialsciencejournal.in ', 'ISSN: 2455-2070 ', 'Received: 19-12-2021, Accepted: 02-01-2022, Published: 19-01-2022 ', 'Volume 8, Issue 1, 2022, Page No. 25-28 ', 'The indigenous badugar of the Nilgiris ', 'H R Sumathi ', 'Assistant Professor, Department of History, Vellalar College for Women, Erode, Tamil Nadu, India ']


In [51]:
# Create collections (a group of logically related documents)
import uuid

collection = client.get_or_create_collection("indigenous_badagar")
collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],
    documents=chunks,
    metadatas=[{"line_number": line} for line in range(len(chunks))]
)

In [52]:
results = collection.query(
    query_texts=[
        "Who are badagas?",
        "When di they come to Nilgiris?"
    ],
    n_results=5
)

results

{'ids': [['fb0ab2d9-a896-4705-8612-9bbddbff8a3d',
   '11391347-85a8-4627-96d6-2c515a5c7cc6',
   'fca285cb-e59a-4c86-b8db-dd14710e5843',
   'ea94c6db-f618-4141-a70b-1d56fc837c25',
   '3bc933aa-1658-483a-bafe-280c74664680'],
  ['f6b5e455-440c-469d-818d-ec68b3e91582',
   '12d3c304-8b92-4069-9b88-64cab36d1af7',
   '7de079e7-2016-406b-b760-af81305feb01',
   'a7f9e172-fe8a-4706-91f4-8fac51d03288',
   'ad16da22-0684-4683-9b14-2680e1a1cb4e']],
 'embeddings': None,
 'documents': [['4. The word Badaga means “people of the  North and ',
   '4. The word Badaga means “people of the  North and ',
   '4. The word Badaga means “people of the  North and ',
   "earlier proceedings connected to the inclusion of 'Badaga ",
   "earlier proceedings connected to the inclusion of 'Badaga "],
  ['the Nilgiri ',
   'the Nilgiri ',
   'the Nilgiri ',
   '2. They are supposed to have migrated to the Nilgiris from ',
   '2. They are supposed to have migrated to the Nilgiris from ']],
 'uris': None,
 'included': ['

In [53]:
for i, query_results in enumerate(results["documents"]):
    print(f"\nQuery: {i}")
    print("\n".join(query_results))


Query: 0
4. The word Badaga means “people of the  North and 
4. The word Badaga means “people of the  North and 
4. The word Badaga means “people of the  North and 
earlier proceedings connected to the inclusion of 'Badaga 
earlier proceedings connected to the inclusion of 'Badaga 

Query: 1
the Nilgiri 
the Nilgiri 
the Nilgiri 
2. They are supposed to have migrated to the Nilgiris from 
2. They are supposed to have migrated to the Nilgiris from 


# Use OpenAI Integration

In [54]:
!pip install -U langchain_chroma

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [55]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os

load_dotenv()
embedding_model = OpenAIEmbeddings(
    model=os.getenv("OPENAI_EMBEDDING_MODEL"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY")
)

In [56]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="indigineous_badagar",
    embedding_function=embedding_model,
    persist_directory="./chroma_db_data"       # optional for in-memory
)

In [59]:
# Add and retrieve documents

import uuid
from langchain_core.documents import Document

documents = []
for i, chunk in enumerate(chunks):
    documents.append(Document(
        page_content=chunk,
        metadata={"page_number": i},
        id=str(uuid.uuid4())
    ))

In [60]:
vectorstore.add_documents(documents=documents)

['76cf88b3-21a9-4157-b96c-076007e1e0ac',
 '960d72d4-e827-46de-ba27-8035ef0fb865',
 'd7f8ed3c-2744-40ff-9a22-11ce5d0d9092',
 'a884c541-2ea1-4eb3-a544-d88d3acdde77',
 '073c0d2f-7705-4410-a2c5-22b306785b15',
 '8871ccbb-f122-44f1-b605-cc2a2b1380e0',
 'ba96adcd-496f-41d8-a0da-3ff4d28939fc',
 '9099f222-3f54-4721-9a79-500556716d9f',
 '5fdf1c05-f7fc-45c2-b58d-85b37ca84599',
 'c4909753-950f-45ac-8fb7-80255c8027d7',
 '31104d41-92c4-4db7-bf3f-6913c3cfb00f',
 '381fe2bd-aaa7-45a3-af5c-1083501fa98e',
 'c604d526-ee39-4f11-b2bc-88eb102ec637',
 'a13dc16b-bcc2-4d49-92e3-914031234de6',
 '879b0d9a-856c-4ff0-a63b-cca69f57bebd',
 '15793647-c0bb-4fc6-8d2c-6a7dee70e045',
 '227510ef-6738-46b8-b04f-a9160b1a984c',
 '1ecc91ca-1a52-45c9-bf3c-31cdb8f4112b',
 'b56d928a-ec24-47de-9c40-732e012d302b',
 'cb356a41-7d2f-4551-bc73-0f58544b228f',
 'd6f9b2da-2a6f-4cfd-8630-19d402d54768',
 '463f74b0-745b-439e-8017-ce3d3de8e95e',
 '925c7139-fa4d-49ab-99eb-3afc07d04143',
 '68a5a34e-1e96-4fec-a020-beba6e1e290b',
 '7262aec9-2577-

In [64]:
vectorstore.similarity_search("Who are badagas?")

[Document(id='14dc4dc9-7797-4ba7-8f4f-fd568bd27065', metadata={'page_number': 53}, page_content='present they are called Badugars because of their Badugu '),
 Document(id='8d07d02f-647b-4d27-9bbc-1a9afd1f51b6', metadata={'page_number': 203}, page_content='3. The Badagas or Vadagas are supposed to have come '),
 Document(id='7262aec9-2577-4cc6-ab76-2ebd8ead4d4c', metadata={'page_number': 24}, page_content='supposed the Badugar, the northerners or migrants from Mysore considering Bada, the north in Kanada and hence Badagas, '),
 Document(id='1fe7f6d7-029f-40a2-9d65-7c329f806ce2', metadata={'page_number': 107}, page_content='p.128). So, the natives of Nilgiris call them as Badaga, the ')]

In [63]:
retriever = vectorstore.as_retriever()
retriever.invoke("Who are badagas?")

[Document(id='14dc4dc9-7797-4ba7-8f4f-fd568bd27065', metadata={'page_number': 53}, page_content='present they are called Badugars because of their Badugu '),
 Document(id='8d07d02f-647b-4d27-9bbc-1a9afd1f51b6', metadata={'page_number': 203}, page_content='3. The Badagas or Vadagas are supposed to have come '),
 Document(id='7262aec9-2577-4cc6-ab76-2ebd8ead4d4c', metadata={'page_number': 24}, page_content='supposed the Badugar, the northerners or migrants from Mysore considering Bada, the north in Kanada and hence Badagas, '),
 Document(id='1fe7f6d7-029f-40a2-9d65-7c329f806ce2', metadata={'page_number': 107}, page_content='p.128). So, the natives of Nilgiris call them as Badaga, the ')]